In [6]:
from langchain_core.documents import Document
import os
from langchain_community.document_loaders.pdf import PyPDFLoader


/var/folders/x4/wn_b4wfj2zv5c4xdgj44f8r00000gn/T/ipykernel_79335/50779311.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.pdf import PyPDFLoader


## INGESTION PIPELINE

In [7]:
def load_all_pdfs():
    folder_path = "data/pdfs"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            pdf_path = os.path.join(folder_path, filename)
            loader = PyPDFLoader(pdf_path)
            docs = loader.load()  # Loads the document pages into a list
            all_docs.extend(docs)
            num_docs += 1

    print("total pdfs:", num_docs)
    print("total pages:", len(all_docs))
    return all_docs

In [8]:
all_pdf_documents = load_all_pdfs()

total pdfs: 2
total pages: 32


## Chunking

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [10]:
def split_docs(documents, chunk_size=500, chunk_overlap=50):
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [11]:
chunks = split_docs(all_pdf_documents)

## Embedding

In [12]:
from sentence_transformers import SentenceTransformer

In [13]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        
        self.model_name = model_name
        self.model = SentenceTransformer(self.model_name)

    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar = True)
        return embeddings

In [14]:
embedding_manager = EmbeddingManager()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## VECTOR STORE

In [15]:
import chromadb
import uuid

In [16]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.persist_directory = persist_directory
        self.collection_name = collection_name
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok = True)
        self.client = chromadb.PersistentClient(path=self.persist_directory)
        self.collection = self.client.get_or_create_collection(
            name = self.collection_name,
            metadata={"description":"vector store collection for pdf embeddings in RAG"}
        )

        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())

    def add_documents(self,documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("Num. of docs doesn't match no. of embeddings")

        ids=[]
        all_metadata=[]
        documents_content=[]
        embeddings_list=[]

        for i, (doc,embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids=ids,
                metadatas=all_metadata,
                documents=documents_content,
                embeddings=embeddings_list
            )

In [17]:
vector_store = VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 960


In [18]:
texts=[doc.page_content for doc in chunks]
embedding=embedding_manager.generate_embeddings(texts)
vector_store.add_documents(chunks,embedding)

Batches:   0%|          | 0/10 [00:00<?, ?it/s]


## Retrieval Pipeline

In [19]:
from sklearn.metrics.pairwise import cosine_similarity

In [20]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self, query, top_k=5, score_threshold=0.0):
        query_embeddings= self.embedding_manager.generate_embeddings([query])[0]
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results = top_k
        )

        retrieved_docs = []
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i,(doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "Id" : doc_id,
                        "document" : document,
                        "metadata" : metadata,
                        "distance" : distance,
                        "similarity_score" :similarity_score,
                        "rank" : i+1
                   })
        else:
            print("No documents found !!")

        return retrieved_docs

In [21]:
rag_retriever = RAGRetriever(embedding_manager, vector_store) 

In [22]:
rag_retriever.retrieve("What is RAG?")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[{'Id': 'doc_73fa823d-230f-4675-9a25-cb7ff2521d79',
  'document': 'and speculate on upcoming trends and innovations.\nOur contributions are as follows:\n• In this survey, we present a thorough and systematic\nreview of the state-of-the-art RAG methods, delineating\nits evolution through paradigms including naive RAG,\narXiv:2312.10997v5  [cs.CL]  27 Mar 2024',
  'metadata': {'content_length': 288,
   'page_label': '1',
   'subject': '',
   'keywords': '',
   'doc_index': 11,
   'page': 0,
   'creationdate': '2024-03-28T00:54:45+00:00',
   'trapped': '/False',
   'source': 'data/pdfs/research2.pdf',
   'total_pages': 21,
   'title': '',
   'author': '',
   'moddate': '2024-03-28T00:54:45+00:00',
   'producer': 'pdfTeX-1.40.25',
   'creator': 'LaTeX with hyperref',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5'},
  'distance': 0.46291500329971313,
  'similarity_score': 0.5370849967002869,
  'rank': 1},
 {'Id': 'doc_312643da-

LLM INTERGRATION

In [23]:
from langchain_google_genai import ChatGoogleGenerativeAI

import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    max_output_tokens=1024,
)

In [24]:
def generate_output(query, rag_retriever, llm, top_k=3):
    results = rag_retriever.retrieve(query)
    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("Found no relevant context")

    prompt = f"""use given context to generate the answer for the query
Context : {context}
Query : {query}"""

    response = llm.invoke(prompt)
    
    # If Gemini returns a list of content blocks, extract the text
    if isinstance(response.content, list):
        return "".join([part.get("text", "") for part in response.content if isinstance(part, dict)])
    
    return response.content

In [25]:
answer = generate_output("what is encoder-decoder?", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [26]:
print(answer)

Based on the provided context, the text describes the **Decoder** component of the model:

* **Structure:** It is composed of a stack of $N = 6$ identical layers.
* **Sub-layers:** In addition to the two sub-layers found in each encoder layer, the decoder inserts a third sub-layer that performs multi-head attention over the output of the encoder stack. 
* **Connections & Normalization:** Similar to the encoder, it employs residual connections around each of the sub-layers, followed by layer normalization.
* **Outputs:** The self-attention layers produce outputs of dimension $d_{\text{model}} = 512$.
* **Other details:** Positional encodings are used in both the encoder and decoder stacks, and the base model uses a dropout rate of $P_{\text{drop}} = 0.1$. 

*(Note: The provided context describes the architecture and components of the decoder and its relation to the encoder, but it does not explicitly provide a general definition of "encoder-decoder" beyond these structural details.)*


Note: you may need to restart the kernel to use updated packages.
